In [1]:
import sys
print("Using Python version:", sys.version)

Using Python version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [2]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import xml.etree.ElementTree as ET 
from IPython.display import display, Markdown

import requests
from openai import OpenAI

In [ ]:
## Configuration and Client set-up

# Defining the PuBMed webservice endpoints
PUBMED_SEARCH_URL = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi")  # search engine endpoint, which returns a list of PubMed IDs

# The data retreival endpoint, takes a list of PMIDs, 
# it returns the actual article metadata, including paper titles, publication dates, and full abstract texts in XML or text format
PUBMED_FETCH_URL = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi")    

NCBI_EMAIL = "manjunathrampure.co@gmail.com"

# Dynamically selecting the AI model
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

#Instantiates the primary OpenAI() client object from the openai SDK library, which manages connections, headers, and HTTP calls to OpenAI's servers.
client = OpenAI()

In [4]:
# converting the random question into a structured, Boolean-formatted search string that PubMed's search engine can understand.

def create_pubmed_query(question):
    """
    Ask the model to convert a question into a query.
    """
    response = client.responses.create(
        model = MODEL,
        instructions = (
            "Convert the research question into a consise PubMed query. "
            "Use biological keywords, Boolean operators, and synonyms where "
            "helpful. Return only the query"
        ),
        input = question,
    )

    # strip() - removes any leading or trailing whitespace, newlines (\n), or space padding
    return response.output_text.strip()

In [5]:
my_test_query = create_pubmed_query("Is CD19 a better target for B-cell Malignancies?")

In [6]:
print(my_test_query)

("CD19"[Title/Abstract] OR "B-lymphocyte antigen CD19"[Title/Abstract]) AND ("B-cell malignanc*"[Title/Abstract] OR "B-cell neoplasm*"[Title/Abstract] OR "B-cell lymphoma*"[Title/Abstract] OR "B-cell leukemia*"[Title/Abstract]) AND (target*[Title/Abstract] OR therapeutic*[Title/Abstract] OR immunotherap*[Title/Abstract])


In [7]:
## search pubmed 

def search_pubmed(query, maximum_results = 5):
    """
    Search PubMed and return the matching PubMed IDs
    """
    # parentheses to your (query) can group your search query together so that the filter applies to the entire query
    # pubmed acts as an database search engine, with special tags in [] 
    search_query = f"({query}) AND hasabstract[Filter]"

    parameters = {
        "db": "pubmed",
        "term": search_query,
        "retmax": maximum_results,
        "retmode": "json",
        "sort": "relevance",
        "tool": "hackbio_pubmed_course", #NCBI usage requirements that identify your application and provide a contact email
        "email": NCBI_EMAIL, 
    }

    response = requests.get(
        PUBMED_SEARCH_URL,
        params = parameters,
        timeout = 30
    )
    
    response.raise_for_status()

    # parsing the returned JSON data and extracts the list of PubMed ID strings stored inside th esearch result
    return response.json()["esearchresult"]["idlist"]

In [8]:
# extracting the PMIDs by using the search_pubmed function
my_test_pids = search_pubmed(my_test_query)

In [9]:
# extracted PMIDs are
print(my_test_pids)

['35012754', '37055515', '24667955', '36867678', '37294963']


In [10]:
## Fetch abstract
# this block returns the pubmed ID numbers, into structured python dictionaries containing the title, publication year, abstract text, and direct web link for each article.


# defining a small helper function, which cleans up the text extracted from an XML element
def element_text(element):
    """
    Collect all text contained inside an XML element.
    """

    if element is None:
        return ""

    return " ".join("".join(element.itertext()).split())

# main function
def fetch_abstracts(pubmed_ids):
    """
    Retrieve article information for a list of PubMeb IDs.
    """

    # input check and parameter setup
    if not pubmed_ids:
        return []

    parameters = {
        "db": "pubmed",
        "id": ",".join(pubmed_ids),
        "retmode": "xml",
        "tool": "hackbio_pubmed_course",
        "email": NCBI_EMAIL,
    }

    # make the HTTP request and parse the XML
    response = requests.get(
        PUBMED_FETCH_URL,
        params = parameters,
        timeout = 30
    )

    # the returned XML response is parsed into an ElementTree root node(root) so Python can search through XML tags
    root = ET.fromstring(response.content)
    articles = []

    # loop through articles and extract the fields
    for record in root.findall(".//PubmedArticle"):
        citation = record.find("MedlineCitation")
        article = citation.find("Article")

        # extracting title and PMIDs
        pmid = citation.findtext("PMID", default = "")
        title = element_text(article.find("ArticleTitle"))

        # reconstructing structured abstracts
        # scientific abstracts are often split into labeled sections(BACKGROUND, METHODS, RESULTS, CONCLUSION)
        abstract_parts = []
        for section in article.findall("./Abstract/AbstractText"):
            text = element_text(section)
            label = section.attrib.get("Label")

            if label and text:
                abstract_parts.append(f"{label}: {text}")
            elif text:
                abstract_parts.append(text)

        abstract = " ".join(abstract_parts)

        # extracting the Publication Year with Fallbacks
        year = (
            article.findtext("./Journal/JournalIssue/PubDate/Year")
            or article.findtext("./ArticleDate/Year")
            or "Year not available"
        )

        # Build the result dictionary
        if abstract:
            articles.append(
                {
                    "pmid": pmid,
                    "title": title,
                    "year": year,
                    "abstract": abstract,
                    "link": f"https://pubmed.ncbi.nlm.nih.gov./{pmid}",
                }
            )

    return articles

In [11]:
# fetching the abstracts from the given PMIDs 
my_test_articles = fetch_abstracts(my_test_pids)

In [12]:
# extracted abstracts are
print(my_test_articles)

[{'pmid': '35012754', 'title': 'CAR T-cell therapy for B-cell lymphoma.', 'year': '2022', 'abstract': "Chimeric antigen receptor-modified (CAR) T-cell therapy targeting CD19 has revolutionized the treatment of relapsed or refractory B-cell lymphomas. Based on unprecedented response rates and durability of response in high risk B-cell lymphoma patients, anti-CD19 CAR T-cell therapy was rapidly approved by the FDA for a variety of lymphoma subtypes. Anti-CD19 CAR T-cell therapy is now considered standard of care for patients with relapsed or refractory (R/R) aggressive non-Hodgkin's Lymphoma (NHL) after 2 or more lines of therapy. Three second-generation anti-CD19 CAR T-cell products have been FDA approved for R/R aggressive B-cell lymphoma and FDA approval has been obtained for Mantle Cell Lymphoma and Follicular lymphoma as well. This has ensured broad access to CAR T-cell therapy for patients with NHL and new real-world trials have helped confirm feasibility of CAR T-cell therapy for 

In [13]:
# reviewing abstacts 

def review_abstracts(question, query, articles):
    """
    Ask the model to review the retrieved abstracts.
    """ 

    article_text = ""

    # article formatting loop
    for number, article in enumerate(articles, start = 1):
        article_text += f""" 
        ARTICLE {number}
        PMID: {article["pmid"]}
        Title: {article["title"]}
        Year: {article["year"]}
        Link: {article["link"]}
        Abstract: {article["abstract"]}
        """

    # defining system instructions/Prompt Engineering
    instructions = """
        You are assisting a biologist with a small PubMed abstract review.

        Use only the supplied abstracts. Do not invent findings, methods, or references. If information is missing, say so.

        Prepare a report containing:

        1. Research question
        2. PubMed query
        3. A short summary of each article
        4. Common findinds
        5. Important differences
        6. Limitations of the evidence
        7. References with PubMed IDs and links

        Place the relevant PMID after every article-level summary.
        Clearly state that the report is based only on abstracts.  
    """

    response = client.responses.create(
        model = MODEL,
        instructions = instructions,
        input = f""" 
                Research question: {question}
                PubMed query: {query}

            Retrieved articles:
            {article_text}
        """,
    )

    return response.output_text

In [14]:
# generating a test review
my_test_rev = review_abstracts(question = "Is CD19 a better target for B-cell Malignancies?",
                               query = my_test_query,
                               articles = my_test_articles
                               )

In [15]:
# printing my test review
my_test_rev

'# Abstract-based report: Is CD19 a better target for B-cell malignancies?\n\n**Basis of report:** This report is based **only on the supplied PubMed abstracts**. Full-text articles, supplementary data, and independent verification of the reported findings were not reviewed.\n\n## 1. Research question\n\nIs CD19 a better therapeutic target for B-cell malignancies, particularly compared with other potential targets or approaches?\n\nBased on the supplied abstracts, CD19 appears to be a highly validated and clinically effective target for several B-cell malignancies, especially through CD19-directed CAR T-cell therapy. However, the abstracts do not provide a direct, general comparison establishing that CD19 is universally superior to all other targets.\n\n## 2. PubMed query\n\n```text\n("CD19"[Title/Abstract] OR "B-lymphocyte antigen CD19"[Title/Abstract]) AND ("B-cell malignanc*"[Title/Abstract] OR "B-cell neoplasm*"[Title/Abstract] OR "B-cell lymphoma*"[Title/Abstract] OR "B-cell leuke

In [16]:
# Render raw markdown text into a formatted HTML
# display() is a built-in Jupyter function that sends the object directly to the notebook's rich display system.
display(Markdown(my_test_rev)) 

# Abstract-based report: Is CD19 a better target for B-cell malignancies?

**Basis of report:** This report is based **only on the supplied PubMed abstracts**. Full-text articles, supplementary data, and independent verification of the reported findings were not reviewed.

## 1. Research question

Is CD19 a better therapeutic target for B-cell malignancies, particularly compared with other potential targets or approaches?

Based on the supplied abstracts, CD19 appears to be a highly validated and clinically effective target for several B-cell malignancies, especially through CD19-directed CAR T-cell therapy. However, the abstracts do not provide a direct, general comparison establishing that CD19 is universally superior to all other targets.

## 2. PubMed query

```text
("CD19"[Title/Abstract] OR "B-lymphocyte antigen CD19"[Title/Abstract]) AND ("B-cell malignanc*"[Title/Abstract] OR "B-cell neoplasm*"[Title/Abstract] OR "B-cell lymphoma*"[Title/Abstract] OR "B-cell leukemia*"[Title/Abstract]) AND (target*[Title/Abstract] OR therapeutic*[Title/Abstract] OR immunotherap*[Title/Abstract])
```

## 3. Short summary of each article

### Article 1: *CAR T-cell therapy for B-cell lymphoma* — 2022

This review describes anti-CD19 CAR T-cell therapy as having produced unprecedented response rates and durable responses in high-risk B-cell lymphomas. It reports that three second-generation anti-CD19 CAR T-cell products have received FDA approval for relapsed or refractory aggressive B-cell lymphoma, with additional approvals for mantle cell and follicular lymphoma. The abstract also emphasizes broader real-world feasibility and the need to address treatment failure mechanisms after CD19 CAR T-cell therapy. **PMID: 35012754**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/35012754/)

### Article 2: *Long-term outcomes following CAR T cell therapy: what we know so far* — 2023

This review summarizes long-term efficacy and toxicity data for CAR T cells targeting CD19 or BCMA. It states that CD19-targeted CAR T cells can produce prolonged remissions in B-cell malignancies, with minimal long-term toxicities in many patients, and may be curative for a subset. In contrast, BCMA-targeted CAR T-cell remissions are described as generally shorter, although also associated with limited long-term toxicities. The abstract identifies depth of initial response, malignancy characteristics, peak CAR-cell levels, and lymphodepleting chemotherapy as factors associated with long-term remission. **PMID: 37055515**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/37055515/)

### Article 3: *CD19-CAR trials* — 2014

This review describes CD19 as a B-lineage-specific transmembrane glycoprotein expressed on more than 95% of B-cell malignancies, making it an attractive target for CAR-based immune therapy. Early phase 1 trials in relapsed B-cell malignancies, including many chemotherapy-refractory cases, produced major responses, particularly in chronic lymphocytic leukemia and acute lymphoblastic leukemia. The abstract also reports systemic inflammatory reactions that could be life-threatening but were usually reversible with appropriate medical management. **PMID: 24667955**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/24667955/)

### Article 4: *CD19 CAR antigen engagement mechanisms and affinity tuning* — 2023

This study examines how the affinity of CAR binding domains affects CD19 CAR T-cell activity. Structural analyses and molecular simulations were used to create binders with different affinities. The resulting CAR T cells differed in the antigen density required to trigger cytolysis and in their tendency to induce trogocytosis. The findings suggest that structural information and affinity tuning may allow CD19 CAR T-cell performance to be adapted to tumors with different levels of antigen expression. **PMID: 36867678**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/36867678/)

### Article 5: *CAR T-cell therapy in large B cell lymphoma* — 2023

This review summarizes clinical evidence for CD19 CAR T-cell therapy in large B-cell lymphoma. It describes regulatory approvals of three CD19 CAR T-cell products in the third-line setting and subsequent investigation in earlier treatment lines. It also reports promising studies in patients with primary or secondary central nervous system lymphoma, a population excluded from some earlier trials. **PMID: 37294963**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/37294963/)

## 4. Common findings

Across the abstracts, the following themes recur:

- **Strong clinical activity:** CD19 CAR T-cell therapy is associated with major or high response rates in relapsed or refractory B-cell malignancies.
- **Durable benefit:** Several abstracts describe prolonged or durable remissions, with one review stating that CD19 CAR T cells may be curative for a subset of patients.
- **Broad disease applicability:** CD19-directed therapy has been used or investigated in B-cell lymphomas, large B-cell lymphoma, chronic lymphocytic leukemia, and acute lymphoblastic leukemia.
- **Regulatory validation:** Multiple CD19 CAR T-cell products have received FDA approval, supporting CD19 as a clinically established target.
- **Lineage-restricted expression:** The abstracts emphasize that CD19 is associated with the B-cell lineage and is expressed on a large proportion of B-cell malignancies.
- **Important toxicity considerations:** Treatment can cause serious systemic inflammatory reactions, although the abstract describing early trials states that these were usually reversible with appropriate management.
- **Treatment failure remains relevant:** The abstracts identify the need to understand and overcome mechanisms of CAR T-cell failure.
- **CAR design matters:** Antigen density, binder affinity, CAR expansion or persistence, and other treatment-related factors may influence efficacy and toxicity.

## 5. Important differences

### Disease populations

- Article 1 covers a broad range of B-cell lymphomas, including aggressive and indolent lymphomas, mantle cell lymphoma, and follicular lymphoma.
- Article 5 focuses specifically on large B-cell lymphoma and discusses treatment in earlier lines and in central nervous system lymphoma.
- Article 3 emphasizes early trials in relapsed B-cell malignancies, particularly chronic lymphocytic leukemia and acute lymphoblastic leukemia.
- Article 2 considers B-cell malignancies more broadly and compares CD19-targeted CAR T cells with BCMA-targeted CAR T cells, primarily in the context of different disease groups.

### Type of evidence

- Articles 1, 2, 3, and 5 are reviews of clinical experience or clinical trial data.
- Article 4 is primarily a mechanistic and structural study involving CAR binding affinity, antigen density, cytolysis, and trogocytosis.
- The abstracts do not indicate that any of these articles is a systematic meta-analysis directly comparing CD19 with all alternative targets.

### Comparison with other targets

- Article 2 provides the clearest comparison, reporting that CD19-targeted CAR T cells produce more prolonged remissions than BCMA-targeted CAR T cells in the summarized evidence.
- However, the comparison involves different target antigens and different disease contexts, so it does not establish that CD19 is intrinsically superior in every B-cell malignancy.
- The other abstracts mainly evaluate or describe CD19 without direct comparisons with alternative B-cell targets.

### Stage of treatment and clinical setting

- The abstracts discuss CD19 CAR T-cell use after multiple prior lines of therapy, in second-line treatment, and potentially earlier.
- Article 5 additionally discusses patients with central nervous system involvement, whereas earlier trials often excluded such patients.
- These differences make response and safety findings difficult to compare directly across articles.

### Mechanisms and optimization

- Article 4 focuses on optimizing the CAR itself through affinity tuning and antigen-density sensitivity.
- Articles 1 and 2 focus more on long-term clinical outcomes, remission durability, toxicity, and treatment failure.
- Article 3 emphasizes the early clinical validation of the CD19 target.

## 6. Limitations of the evidence

- **Abstract-only evidence:** The conclusions are limited to information in the supplied abstracts. Details such as sample sizes, response percentages, confidence intervals, follow-up duration, patient selection, and statistical analyses are unavailable.
- **No direct universal comparison:** The abstracts do not provide a head-to-head comparison of CD19 with all other potential targets for B-cell malignancies.
- **Different diseases and targets:** The CD19-versus-BCMA comparison in Article 2 concerns different malignancy contexts, which limits interpretation of target superiority.
- **Predominance of review articles:** Four of the five supplied abstracts are reviews or broad summaries, rather than clearly identified comparative clinical trials.
- **Potential selection and publication bias:** The abstracts emphasize successful development, approvals, and clinical responses; they do not provide enough information to assess unpublished negative results or the full range of treatment outcomes.
- **Incomplete toxicity information:** Serious toxicities are mentioned, but the supplied abstracts do not provide comparative rates or a full description of acute and late adverse events.
- **Limited information on resistance:** Treatment failure is acknowledged, but the abstracts do not quantify antigen loss, inadequate persistence, tumor microenvironment effects, or other mechanisms.
- **Unclear generalizability:** The abstracts do not provide enough detail to determine how outcomes vary by age, comorbidity, disease burden, prior treatment, antigen density, or access to specialized CAR T-cell care.
- **No complete assessment of non-CAR approaches:** The supplied evidence focuses largely on CD19 CAR T-cell therapy and does not establish whether CD19 is superior for antibody-based therapies, bispecific antibodies, vaccines, or other therapeutic platforms.

## Overall conclusion

The supplied abstracts support CD19 as a **strongly validated and clinically effective target** for multiple B-cell malignancies. Its lineage-associated expression, major response rates, durable remissions, regulatory approvals, and applicability across several B-cell diseases make it an important therapeutic target.

Nevertheless, the evidence provided does **not demonstrate that CD19 is universally the “better” target** for all B-cell malignancies. The strongest comparative statement comes from the review contrasting CD19 with BCMA, which reports more prolonged remissions with CD19-targeted CAR T cells; however, this comparison is affected by differences in disease type and clinical context. Based only on these abstracts, CD19 is best characterized as a **well-established and highly effective target**, rather than definitively superior to every alternative target.

## 7. References

1. *CAR T-cell therapy for B-cell lymphoma.* 2022. **PMID: 35012754.**  
   https://pubmed.ncbi.nlm.nih.gov/35012754/

2. *Long-term outcomes following CAR T cell therapy: what we know so far.* 2023. **PMID: 37055515.**  
   https://pubmed.ncbi.nlm.nih.gov/37055515/

3. *CD19-CAR trials.* 2014. **PMID: 24667955.**  
   https://pubmed.ncbi.nlm.nih.gov/24667955/

4. *CD19 CAR antigen engagement mechanisms and affinity tuning.* 2023. **PMID: 36867678.**  
   https://pubmed.ncbi.nlm.nih.gov/36867678/

5. *CAR T-cell therapy in large B cell lymphoma.* 2023. **PMID: 37294963.**  
   https://pubmed.ncbi.nlm.nih.gov/37294963/

In [ ]:
## Optional; If you want to save your literature reviews into a markdown file
# Later you can use it to convert into PDF as well

markdown_filename = "CD19_literature_review.md"
with open(markdown_filename, "w", encoding="utf-8") as f:
    f.write(my_test_rev)

print(f"Report saved as raw Markdown to {markdown_filename}")

Report saved as raw Markdown to CD19_literature_review.md
